# 13-bob. Adversarial mashinali o'qitish
## Kengaytirilgan amaliy ish: a'zolikni aniqlash hujumi

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/13-bob/13-3_azolikni-aniqlash-hujumi.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 13-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dasturda 1000 ta sun'iy yozuvning yarmi modelni o'qitishda ishlatiladi ("a'zolar"), qolgan yarmi esa ishlatilmaydi ("begonalar"). Buzg'unchi faqat modelning to'g'ri sinf uchun ishonch qiymatini ko'radi va shu qiymat bo'yicha yozuvning a'zo ekanini taxmin qiladi. Hujum sifati ROC egri chizig'i ostidagi yuza bilan baholanadi: 0,5 — tasodifiy taxmin, 1 — mukammal hujum. Hujum uchta tasodifiy o'rmon modeliga qarshi sinaladi: cheklovsiz va ikki xil usulda cheklangan.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [2]:
# 1. Sun'iy "mijozlar" ma'lumotlari: modelni o'qitish uchun 500 ta, boshqasi 500 ta (a'zo emas)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=5, flip_y=0.15, random_state=13)
X_azo, y_azo, X_begona, y_begona = X[:500], y[:500], X[500:], y[500:]

def hujum(model):
    """Buzg'unchi faqat modelning to'g'ri sinf uchun ishonchini ko'radi: ishonch yuqori bo'lsa - 'a'zo'."""
    ishonch_azo = model.predict_proba(X_azo)[np.arange(500), y_azo]
    ishonch_begona = model.predict_proba(X_begona)[np.arange(500), y_begona]
    ball = np.r_[ishonch_azo, ishonch_begona]
    haqiqat = np.r_[np.ones(500), np.zeros(500)]
    return roc_auc_score(haqiqat, ball), ishonch_azo.mean(), ishonch_begona.mean()

print("{:<36}{:>13}{:>10}{:>11}".format("Model", "o'qitishdagi", "sinovdagi", "hujum AUC"))
for nom, model in [("Chuqur daraxtlar (cheklovsiz)", RandomForestClassifier(200, random_state=0)),
                   ("Cheklangan (max_depth=4)", RandomForestClassifier(200, max_depth=4, random_state=0)),
                   ("Cheklangan (min_samples_leaf=20)", RandomForestClassifier(200, min_samples_leaf=20, random_state=0))]:
    model.fit(X_azo, y_azo)
    auc, i_azo, i_beg = hujum(model)
    tog_o = model.score(X_azo, y_azo); tog_s = model.score(X_begona, y_begona)
    print(f"{nom:<36}{tog_o:>13.2f}{tog_s:>10.2f}{auc:>11.2f}   (o'rtacha ishonch: a'zo {i_azo:.2f}, begona {i_beg:.2f})")

Model                                o'qitishdagi sinovdagi  hujum AUC


Chuqur daraxtlar (cheklovsiz)                1.00      0.80       0.93   (o'rtacha ishonch: a'zo 0.86, begona 0.63)


Cheklangan (max_depth=4)                     0.88      0.75       0.63   (o'rtacha ishonch: a'zo 0.63, begona 0.58)


Cheklangan (min_samples_leaf=20)             0.85      0.73       0.61   (o'rtacha ishonch: a'zo 0.62, begona 0.58)


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Model                                o'qitishdagi sinovdagi  hujum AUC
Chuqur daraxtlar (cheklovsiz)                1.00      0.80       0.93   (o'rtacha ishonch: a'zo 0.86, begona 0.63)
Cheklangan (max_depth=4)                     0.88      0.75       0.63   (o'rtacha ishonch: a'zo 0.63, begona 0.58)
Cheklangan (min_samples_leaf=20)             0.85      0.73       0.61   (o'rtacha ishonch: a'zo 0.62, begona 0.58)
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 13-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.